# Indonesian Twitter Sarcasm Detection — Transformer Baselines

Eksperimen baseline dari **IEEE Access 2024** menggunakan dataset Twitter yang sudah dipreproses (`twitter_ready`).

## Cara pakai
1. Enable GPU: **Notebook Settings → Accelerator → GPU T4 x2** (BUKAN P100 — P100 tidak didukung build PyTorch saat ini)
2. (Opsional) Di Cell 1, edit `MODELS_TO_RUN` kalau mau jalankan subset model tertentu
3. **Run All** — semua model dijalankan otomatis secara berurutan

> **GPU per-model (Opsi B):** XLM-R otomatis pakai 1 GPU (batch 32, hindari OOM/instabilitas), model lain pakai 2 GPU (batch 64, cocok angka paper). Diatur di Cell 5.

> Semua 8 model bisa dijalankan dalam 1 sesi. Hasil tiap model disimpan ke `all_results.json` setelah selesai, jadi kalau sesi putus di tengah hasil sebelumnya tetap tersimpan.

In [ ]:
# ============================================================
# CELL 1 — KONFIGURASI
# ============================================================

ALL_MODELS = {
    "1_indobert_base_indonlu":  "indobenchmark/indobert-base-p1",
    "2_indobert_large_indonlu": "indobenchmark/indobert-large-p1",
    "3_indobert_base_indolem":  "indolem/indobert-base-uncased",
    "4_mbert_base":             "google-bert/bert-base-multilingual-cased",
    "5_xlmr_base":              "FacebookAI/xlm-roberta-base",
    "6_xlmr_large":             "FacebookAI/xlm-roberta-large",
    "7_nusabert_base":          "w11wo/nusabert-base",
    "8_nusabert_large":         "w11wo/nusabert-large",
}

# Mau jalankan semua? Biarkan None.
# Mau subset? Contoh: ["1_indobert_base_indonlu", "5_xlmr_base"]
MODELS_TO_RUN = None

# HuggingFace token — diperlukan untuk model private (NusaBERT).
# Simpan sebagai Kaggle Secret dengan key "HF_TOKEN", atau isi langsung di sini.
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = ""  # isi manual kalau tidak pakai Kaggle Secrets

# ── Resolve daftar model ─────────────────────────────────────
if MODELS_TO_RUN is None:
    MODELS_TO_RUN = list(ALL_MODELS.keys())

print(f"Model yang akan dijalankan ({len(MODELS_TO_RUN)}):")
for key in MODELS_TO_RUN:
    print(f"  {key:30s} → {ALL_MODELS[key]}")
print(f"HF_TOKEN : {'set ✓' if HF_TOKEN else 'kosong (model publik saja)'}")

In [ ]:
# ============================================================
# CELL 2 — Setup environment (clone repo, set working dir)
# ============================================================
import os, sys

# Akselerator: gunakan GPU **T4 x2** (BUKAN P100 -- P100/Pascal sm_60 tidak
# didukung build PyTorch saat ini -> CUDA "no kernel image" error).
#
# Pengaturan GPU per-model (Opsi B) dilakukan di CELL 5, BUKAN di sini:
#   - XLM-R   -> 1 GPU  (batch efektif 32) : hindari OOM (large) + instabilitas
#   - lainnya -> 2 GPU  (batch efektif 64) : cocok dengan angka baseline paper

REPO_URL = "https://github.com/audihus/Sarcasm-Detection.git"
REPO_DIR = "/kaggle/working/sarcasm"
WORK_DIR = f"{REPO_DIR}/id_sarcasm"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
    print("Repo cloned.")
else:
    print("Repo sudah ada, pull latest...")
    os.system(f"git -C {REPO_DIR} pull")

os.chdir(WORK_DIR)
sys.path.append(WORK_DIR)

print(f"Working dir : {os.getcwd()}")

In [ ]:
# ============================================================
# CELL 3 — Install dependencies (PIN versi)
# ============================================================
import subprocess

# Pin ke versi tetap paling dekat dengan era paper yang MASIH didukung script ini.
# Catatan: run_classification.py memakai API baru (processing_class, eval_strategy)
# yang baru ada sejak transformers 4.46 -> kita TIDAK bisa turun ke ~4.41 (era paper,
# Juni 2024) tanpa merombak script. transformers 4.46.x = versi paling awal yang
# mendukung script ini, sekaligus paling dekat ke era paper. torch dibiarkan bawaan
# Kaggle (jangan di-pin -> hindari CUDA mismatch seperti error P100 kemarin).
pinned = [
    "transformers==4.46.3",
    "datasets==3.1.0",
    "evaluate==0.4.3",
    "accelerate==1.1.1",
]
other = ["peft", "scikit-learn", "sentencepiece"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pinned + other, check=True)

import transformers, datasets, accelerate
print("Dependencies installed.")
print("transformers :", transformers.__version__)
print("datasets     :", datasets.__version__)
print("accelerate   :", accelerate.__version__)
os.system("nvidia-smi")

In [ ]:
# ============================================================
# CELL 4 — Verifikasi script dan data
# ============================================================
import pandas as pd

SCRIPT  = "scripts/run_classification.py"
TRAIN   = "preprocessed_data/twitter_ready/train.csv"
VAL     = "preprocessed_data/twitter_ready/validation.csv"
TEST    = "preprocessed_data/twitter_ready/test.csv"
SUMMARY = "/kaggle/working/all_results.json"

assert os.path.exists(SCRIPT), f"Script tidak ditemukan: {SCRIPT}"
print(f"Script : {SCRIPT} ✓")

for name, path in [("Train", TRAIN), ("Validation", VAL), ("Test", TEST)]:
    assert os.path.exists(path), f"{name} file tidak ditemukan: {path}"
    df = pd.read_csv(path)
    counts = df["label"].value_counts().sort_index().to_dict()
    print(f"{name:12s}: {len(df):5d} samples | labels: {counts}")

In [ ]:
# ============================================================
# CELL 5 — Training loop (semua model berurutan)
# ============================================================
import json, shutil, sys, time

def load_summary():
    if os.path.exists(SUMMARY):
        with open(SUMMARY) as f:
            return json.load(f)
    return []

def save_summary(data):
    with open(SUMMARY, "w") as f:
        json.dump(data, f, indent=2)

def run_model(model_key, model_id):
    output_dir = f"outputs/{model_key}-twitter"
    shutil.rmtree(output_dir, ignore_errors=True)
    os.makedirs(output_dir, exist_ok=True)

    token_flag = f" --token {HF_TOKEN}" if HF_TOKEN else ""

    cmd = (
        f'"{sys.executable}" {SCRIPT}'
        f" --model_name_or_path {model_id}"
        f" --train_file {TRAIN}"
        f" --validation_file {VAL}"
        f" --test_file {TEST}"
        f" --text_column_names content"
        f" --label_column_name label"
        f" --shuffle_train_dataset"
        f" --metric_name f1"
        f" --max_seq_length 128"
        f" --per_device_train_batch_size 32"
        f" --per_device_eval_batch_size 64"
        f" --learning_rate 1e-5"
        f" --lr_scheduler_type cosine"
        f" --weight_decay 0.03"
        f" --label_smoothing_factor 0.0"
        f" --num_train_epochs 100"
        f" --do_train --do_eval --do_predict"
        f" --output_dir {output_dir}"
        f" --save_strategy epoch"
        f" --eval_strategy epoch"
        f" --logging_strategy epoch"
        f" --load_best_model_at_end"
        f" --metric_for_best_model f1"
        f" --save_total_limit 2"
        f" --seed 42"
        f" --report_to none"
        f" --fp16"
        + token_flag
    )

    # --- Opsi B: atur GPU per-model (subprocess membaca env ini saat start) ---
    #   XLM-R   -> 1 GPU  (batch efektif 32) : hindari OOM (large) + instabilitas kolaps
    #   lainnya -> 2 GPU  (batch efektif 64) : cocok angka baseline paper
    if "xlmr" in model_key:
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    else:
        os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    print(f"  CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}  "
          f"({'1 GPU / batch 32' if 'xlmr' in model_key else '2 GPU / batch 64'})")

    ret = os.system(cmd)
    return ret, output_dir


failed = []
n_total = len(MODELS_TO_RUN)

for i, model_key in enumerate(MODELS_TO_RUN, 1):
    model_id = ALL_MODELS[model_key]

    print("\n" + "=" * 70)
    print(f"[{i}/{n_total}] {model_key}")
    print(f"        {model_id}")
    print("=" * 70)

    t0 = time.time()
    ret, output_dir = run_model(model_key, model_id)
    elapsed = (time.time() - t0) / 60

    eval_file = os.path.join(output_dir, "eval_results.json")
    if ret != 0 or not os.path.exists(eval_file):
        print(f"[GAGAL] return code={ret}, eval_results.json tidak ada")
        failed.append(model_key)
        shutil.rmtree(output_dir, ignore_errors=True)
        continue

    with open(eval_file) as f:
        metrics = json.load(f)

    f1  = metrics.get("eval_f1", float("nan"))
    acc = metrics.get("eval_accuracy", float("nan"))
    print(f"\n  F1={f1:.4f}  Acc={acc:.4f}  ({elapsed:.1f} menit)")

    existing = load_summary()
    existing = [e for e in existing if e.get("model_key") != model_key]
    existing.append({
        "model_key": model_key,
        "model_id":  model_id,
        **{k: round(v, 6) for k, v in metrics.items() if isinstance(v, float)},
    })
    save_summary(existing)
    print(f"  Disimpan ke {SUMMARY}")

    shutil.rmtree(output_dir, ignore_errors=True)
    print(f"  Output dir dihapus (hemat disk)")

    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass

print("\n" + "=" * 70)
print("SELESAI")
if failed:
    print(f"Model yang gagal: {failed}")

In [ ]:
# ============================================================
# CELL 6 — Tabel hasil akhir
# ============================================================
import json, pandas as pd

if not os.path.exists(SUMMARY):
    print("Belum ada hasil.")
else:
    with open(SUMMARY) as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    cols = ["model_key", "eval_f1", "eval_accuracy", "eval_precision", "eval_recall"]
    cols = [c for c in cols if c in df.columns]
    df = df[cols].sort_values("eval_f1", ascending=False).reset_index(drop=True)

    for col in cols[1:]:
        df[col] = df[col].map(lambda x: f"{float(x):.4f}")

    print(df.to_string(index=False))